# 01 - Municipal election results 2011, 2016 and 2021

Goal: one row per municipality per election, with registered voters, votes cast, spoilt votes and turnout, joined
on `muni_code`. We also mark the municipalities whose boundaries changed in 2016.

Reads: `INTERIM / "results_municipal_<year>_clean.csv"` for 2011, 2016 and 2021 (made by the three
`01_clean_municipal` notebooks)
Writes: `INTERIM / "results_lge_2011_2021.csv"`

Run the two setup cells first.

## Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    url = REPO_URL
    try:                                    # private repo: token from Colab Secrets
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        print("No GITHUB_TOKEN in Colab Secrets - cloning will fail for the private repo")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
print(config.describe())

## 1. Load the three clean files

We take the year from the file name, so older versions of the clean files still load correctly.

In [ ]:
YEARS = [2011, 2016, 2021]

frames = []
for year in YEARS:
    path = INTERIM / f"results_municipal_{year}_clean.csv"
    df = clean_columns(pd.read_csv(path)).drop(columns="year", errors="ignore")
    frames.append(df.assign(election=year))
    print(f"{year}: {len(df)} municipalities  <- {path.name}")

raw = pd.concat(frames, ignore_index=True)
raw.head()

### Check: no two elections are copies of each other

If the wrong raw files were cleaned for one year, two clean files can be identical. This happened once (2016 was
a copy of 2021). Two real elections never have exactly the same registered voters everywhere, so if they match,
we stop and redo the cleaning.

In [ ]:
fingerprint = {year: tuple(sorted(g["registeredvoters"])) for year, g in raw.groupby("election")}
for a in YEARS:
    for b in YEARS:
        if a < b:
            assert fingerprint[a] != fingerprint[b], (
                f"results_municipal_{a}_clean.csv and results_municipal_{b}_clean.csv contain the same numbers. "
                f"Check the raw folders (DateGenerated) and re-run 01_clean_municipal_{a} and _{b}."
            )
print("OK: the three elections have different figures.")

## 2. Municipality code and column names

`Municipality` looks like `EC101 - Camdeboo [Graaff-Reinet]`. The code is before the dash. In 2011 the main town is
added in brackets. Metros have letter codes such as `BUF` and `TSH`.

Turnout is kept as a fraction (0 to 1), like everywhere else in the project.

In [ ]:
parts = raw["municipality"].str.extract(r"^\s*([A-Z]{2,3}\d{0,3})\s*-\s*(.+?)\s*$")
assert parts[0].notna().all(), raw.loc[parts[0].isna(), "municipality"].tolist()

results = pd.DataFrame({
    "election":    raw["election"],
    "province":    raw["province"],
    "muni_code":   parts[0],
    "muni_name":   parts[1].str.replace(r"\s*\[.*\]$", "", regex=True),
    "registered":  raw["registeredvoters"],
    "valid_votes": raw["totalvalidvotes"],
    "spoilt":      raw["spoiltvotes"],
})
results["muni_type"]  = np.where(results["muni_code"].str.contains(r"\d"), "local", "metro")
results["votes_cast"] = results["valid_votes"] + results["spoilt"]
results["turnout"]    = results["votes_cast"] / results["registered"]
results.head()

## 3. Mark the 2016 boundary changes

Before the 2016 election many municipalities were merged or re-drawn: 234 became 213. A code can stay the same
while the area gets bigger. For example `EC101` (Camdeboo) took in Ikwezi and Baviaans and almost doubled its
voters. So the same code does not always mean the same area.

`boundary_2016` can be:
- `abolished`: the code exists in 2011 only (it was merged into another municipality)
- `new`: the code exists from 2016 only (created by a merger)
- `resized`: the code exists in both, but registered voters changed by much more than normal growth (2016 / 2011
  outside 0.85 to 1.5)
- `none`: no change found

`comparable_2011` is 1 when `boundary_2016` is `none`, so the area was the same in 2011, 2016 and 2021. Use it for
anything that compares with 2011.

Nationally, registered voters grew about 11% from 2011 to 2016, so a ratio outside 0.85 to 1.5 points to a boundary
change. Small re-draws inside that range are not caught. There were no code changes between 2016 and 2021 (checked
below).

In [ ]:
wide = results.pivot(index="muni_code", columns="election", values="registered")
ratio = wide[2016] / wide[2011]

boundary = pd.Series("none", index=wide.index, name="boundary_2016")
boundary[wide[2011].notna() & wide[2016].isna()] = "abolished"
boundary[wide[2011].isna() & wide[2016].notna()] = "new"
boundary[(ratio < 0.85) | (ratio > 1.5)] = "resized"

results = results.merge(boundary, left_on="muni_code", right_index=True, how="left")
results["comparable_2011"] = results["boundary_2016"].eq("none").astype(int)

print(boundary.value_counts(), "\n")
print("Resized:")
print(pd.DataFrame({"registered_2011": wide[2011], "registered_2016": wide[2016], "ratio": ratio.round(2)})
      [boundary.eq("resized")].sort_values("ratio"))

In [ ]:
# 2016 -> 2021: same set of codes, and no big jumps in registered voters
codes_16 = set(results.query("election == 2016")["muni_code"])
codes_21 = set(results.query("election == 2021")["muni_code"])
print("Codes that differ 2016 vs 2021:", sorted(codes_16 ^ codes_21) or "none")
print((wide[2021] / wide[2016]).describe().round(3))

## 4. COVID flag

The 2021 election (1 November 2021) was held under COVID-19 rules on campaigning and gatherings. `covid_shock` is 1
for 2021 and 0 for 2011 and 2016, so we can tell the COVID dip apart from a longer fall in turnout.

In [ ]:
results["covid_shock"] = (results["election"] == 2021).astype(int)
results.groupby("election")["covid_shock"].first()

## 5. Checks before saving

- 234 municipalities in 2011, and 213 in 2016 and 2021 (district municipalities are not on the ward ballot)
- one row per municipality per election, and no missing values
- turnout between 0.2 and 0.9
- national turnout on the ward ballot: about 57.4% (2011), 57.7% (2016) and 45.6% (2021). The IEC's headline
  figures are slightly higher (57.6%, 58.0%, 45.9%) because we count the ward ballot only.
- registered voters match the total of all voting stations in the raw IEC files: 23,654,347 / 26,333,353 / 26,203,560

In [ ]:
counts = results.groupby("election")["muni_code"].nunique()
print(counts, "\n")
assert counts.to_dict() == {2011: 234, 2016: 213, 2021: 213}
assert not results.duplicated(["election", "muni_code"]).any()
assert results.notna().all().all()
assert results["turnout"].between(0.2, 0.9).all()

national = results.groupby("election")[["votes_cast", "registered"]].sum()
national["turnout"] = (national["votes_cast"] / national["registered"]).round(4)
national

In [ ]:
from matplotlib.ticker import PercentFormatter

fig, ax = plt.subplots(figsize=(7, 3.5))
sns.boxplot(data=results, x="election", y="turnout", ax=ax, color="lightsteelblue")
ax.set(title="Municipal turnout by election (ward ballot)", xlabel="", ylabel="Turnout")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
plt.show()

## 6. Save output

In [ ]:
out = results[["election", "province", "muni_code", "muni_name", "muni_type",
               "registered", "valid_votes", "spoilt", "votes_cast", "turnout",
               "boundary_2016", "comparable_2011", "covid_shock"]].sort_values(["election", "muni_code"])

out_path = INTERIM / "results_lge_2011_2021.csv"
out.to_csv(out_path, index=False)
print(f"Saved {out.shape} to: {out_path}")
out.head()